In [ ]:
# 2026-04-21 | Goal: full 8-tilt × 3-seed EDM ablation of tilted score matching on QM9
import random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


In [ ]:
# ── Cell 2: Clone / update repos, set paths ──────────────────────────────
import os, pathlib, subprocess, sys

TAILSEEKER_REPO = 'https://github.com/Aaryan-Patel2/TailSeeker.git'
EDM_REPO        = 'https://github.com/ehoogeboom/e3_diffusion_for_molecules.git'
EDM_COMMIT      = ''   # pin to a SHA for reproducibility — empty = latest main

REPO_ROOT   = '/content/TailSeeker'
EDM_PATH    = '/content/edm'
DATA_ROOT   = '/content/ts_data'
OUTPUT_ROOT = '/content/ts_outputs'

def _sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout: print(r.stdout.rstrip())
    if r.stderr: print(r.stderr.rstrip())
    if check: r.check_returncode()
    return r

# TailSeeker — clone once, pull on subsequent runs
if pathlib.Path(REPO_ROOT, '.git').exists():
    print('TailSeeker: pulling latest ...')
    _sh(f'git -C {REPO_ROOT} pull --ff-only')
else:
    print('TailSeeker: cloning ...')
    _sh(f'git clone {TAILSEEKER_REPO} {REPO_ROOT}')

# EDM — clone once (no upstream changes expected mid-ablation)
if pathlib.Path(EDM_PATH, '.git').exists():
    head = subprocess.check_output(f'git -C {EDM_PATH} rev-parse HEAD', shell=True).decode().strip()
    print(f'EDM: already cloned (HEAD {head[:12]})')
else:
    print('EDM: cloning ...')
    _sh(f'git clone {EDM_REPO} {EDM_PATH}')
    if EDM_COMMIT:
        _sh(f'git -C {EDM_PATH} checkout {EDM_COMMIT}')

for d in [DATA_ROOT, OUTPUT_ROOT]:
    os.makedirs(d, exist_ok=True)

for p in [REPO_ROOT, EDM_PATH]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'Repo:    {REPO_ROOT}')
print(f'EDM:     {EDM_PATH}')
print(f'Data:    {DATA_ROOT}')
print(f'Outputs: {OUTPUT_ROOT}')


In [ ]:
# ── Cell 3: Install dependencies ─────────────────────────────────────────
# Runtime: ~2-4 min on Colab. Skips packages already importable.
import subprocess, sys, importlib

def _pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise RuntimeError(f'pip install failed: {args}')

def _have(*modules):
    return all(importlib.util.find_spec(m) is not None for m in modules)

if not _have('torch_geometric'):
    print('Installing torch-geometric ...')
    _pip('torch-geometric')
else:
    print('torch-geometric: ok')

if not _have('torch_scatter', 'torch_sparse'):
    import torch
    tv     = torch.__version__.split('+')[0]
    cv     = torch.version.cuda
    suffix = f'cu{cv.replace(".", "")}' if cv else 'cpu'
    url    = f'https://data.pyg.org/whl/torch-{tv}+{suffix}.html'
    print(f'Trying scatter/sparse wheels for torch={tv} {suffix} ...')
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'torch-scatter', 'torch-sparse', '-f', url],
        capture_output=True, text=True, timeout=120,
    )
    if r.returncode == 0:
        print('  torch-scatter / torch-sparse installed.')
    else:
        print(f'  No wheels for {tv}+{suffix} — continuing without (PyG 2.x is fine).')
else:
    print('torch-scatter / torch-sparse: ok')

if not _have('rdkit', 'hydra', 'wandb'):
    print('Installing requirements-colab.txt ...')
    _pip('-r', f'{REPO_ROOT}/requirements-colab.txt')
else:
    print('rdkit / hydra / wandb: ok')

if not _have('torch_ema'):
    print('Installing EDM requirements ...')
    _pip('-r', f'{EDM_PATH}/requirements.txt')
else:
    print('torch_ema: ok')

print('Installing TailSeeker (editable) ...')
_pip('-e', REPO_ROOT)

print('\nAll deps ready.')


In [ ]:
# ── Cell 4: Verify imports + device ──────────────────────────────────────
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('CPU only')

from src.losses import term_aggregate, TiltedScoreMatchingLoss, MultiObjectiveTiltedLoss
from src.models.edm_adapter import EDMAdapter
from src.utils import set_seed
print('TailSeeker imports: OK')

import qm9.utils  # verifies EDM is on sys.path
print('EDM imports: OK')

from rdkit import Chem
from rdkit.Chem import QED
print('RDKit: OK')

x = torch.tensor([1.0, 2.0, 3.0])
assert torch.allclose(term_aggregate(x, tilt=0.0), x.mean(), atol=1e-7)
print('term_aggregate sanity: OK')


In [ ]:
# ── Cell 5: Download all data ─────────────────────────────────────────────
# Part A — PyG QM9 (~100 MB, needed by TailSeeker's own dataloader)
# Part B — EDM QM9 raw GDB-9 files (~400 MB) + NPZ processing (~5 min)
# Both are cached after first run; safe to re-run.
import os, sys, urllib.request
from pathlib import Path

# ── Part A: PyG QM9 ───────────────────────────────────────────────────────
from src.data.qm9 import QM9Dataset
print('Part A — PyG QM9:')
for split in ('train', 'val', 'test'):
    ds = QM9Dataset(root=DATA_ROOT, split=split, max_atoms=29)
    print(f'  {split}: {len(ds):,} molecules')

# ── Part B: EDM GDB-9 raw files ───────────────────────────────────────────
print('\nPart B — EDM QM9:')
sys.path.insert(0, EDM_PATH)

QM9_DIR = Path(DATA_ROOT) / 'qm9'
QM9_DIR.mkdir(parents=True, exist_ok=True)

FIGSHARE = {
    'dsgdb9nsd.xyz.tar.bz2': 'https://springernature.figshare.com/ndownloader/files/3195389',
    'uncharacterized.txt':   'https://springernature.figshare.com/ndownloader/files/3195404',
    'atomref.txt':           'https://springernature.figshare.com/ndownloader/files/3195395',
}

def _is_html(p):
    return b'<!doctype' in p.read_bytes()[:400].lower()

def _download(url, dest):
    print(f'  Downloading {dest.name} ...')
    urllib.request.urlretrieve(url, filename=dest)
    if _is_html(dest):
        dest.unlink()
        raise RuntimeError(
            f'Figshare returned an HTML error page for {url}\n'
            'Try again in a few minutes, or download manually.'
        )
    print(f'    {dest.stat().st_size / 1e6:.1f} MB')

for fname, url in FIGSHARE.items():
    dest = QM9_DIR / fname
    if dest.exists() and _is_html(dest):
        print(f'  {fname}: cached file is corrupt (HTML) — deleting and re-downloading.')
        dest.unlink()
    if dest.exists():
        print(f'  {fname}: already present.')
    else:
        _download(url, dest)

# Validate uncharacterized.txt
def _is_int(s):
    try: int(s); return True
    except: return False

excl = QM9_DIR / 'uncharacterized.txt'
with open(excl) as f:
    n_excl = sum(1 for line in f if line.strip() and _is_int(line.split()[0]))

if n_excl != 3054:
    excl.unlink()
    raise RuntimeError(
        f'uncharacterized.txt: got {n_excl} valid entries (expected 3054). '
        'File deleted — re-run this cell.'
    )
print(f'  uncharacterized.txt: {n_excl} excluded molecules — OK.')

# Pre-process into NPZ (EDM's retrieve_dataloaders does this on first call)
train_npz = QM9_DIR / 'train.npz'
if train_npz.exists():
    print('  NPZ already processed — skip.')
else:
    print('  Processing GDB-9 → train/valid/test.npz (~5 min) ...')
    import argparse
    _args = argparse.Namespace(
        dataset='qm9', datadir=str(DATA_ROOT),
        remove_h=False, include_charges=True, filter_n_atoms=None,
        batch_size=64, num_workers=0, n_epochs=1,
    )
    from qm9 import dataset as _qm9_dataset
    _dl, _ = _qm9_dataset.retrieve_dataloaders(_args)
    print(f'  Done. Train batches: {len(_dl["train"])}')

sys.path.remove(EDM_PATH)
print('\nAll data ready.')


In [ ]:
# ── Cell 6: Experiment config — edit here ────────────────────────────────
TILT_VALUES   = [-5, -2, -1, 0, 1, 2, 5, 10]   # 8 tilts
SEEDS         = [0, 1, 2]
MAX_EPOCHS    = 100     # 5 for a smoke-test
BATCH_SIZE    = 64      # reduce to 32 if OOM
NUM_WORKERS   = 2       # 0 on WSL2
EVAL_EVERY    = 1
SKIP_EXISTING = True    # skip runs that already have metrics.csv

print(f'Tilt values : {TILT_VALUES}')
print(f'Seeds       : {SEEDS}')
print(f'Total runs  : {len(TILT_VALUES) * len(SEEDS)}')
print(f'Epochs/run  : {MAX_EPOCHS}')
print(f'Outputs     : {OUTPUT_ROOT}')


In [ ]:
# ── Cell 7: Run ablation loop ─────────────────────────────────────────────
import subprocess, sys, itertools, time, os
from pathlib import Path

manifest_path = Path(OUTPUT_ROOT) / 'run_manifest.csv'
if not manifest_path.exists():
    manifest_path.write_text('tilt,seed,status,start_time,elapsed_s\n')

for tilt, seed in itertools.product(TILT_VALUES, SEEDS):
    run_tag = f'tilt{tilt:+.1f}_seed{seed}'
    run_dir = Path(OUTPUT_ROOT) / run_tag

    if SKIP_EXISTING and (run_dir / 'metrics.csv').exists():
        print(f'  skip  {run_tag}')
        continue

    print(f'\n{"="*60}\n  RUNNING  {run_tag}\n{"="*60}')
    t0 = time.time()

    cmd = [
        sys.executable,
        os.path.join(REPO_ROOT, 'scripts', 'run_edm_ablation.py'),
        '--config-name', 'edm_ablation',
        f'loss.tilt={float(tilt)}',
        f'seed={seed}',
        f'training.max_epochs={MAX_EPOCHS}',
        f'training.batch_size={BATCH_SIZE}',
        f'data.num_workers={NUM_WORKERS}',
        f'ablation.eval_every={EVAL_EVERY}',
        f'edm.repo_path={EDM_PATH}',
        f'data.root={DATA_ROOT}',
        f'output.root={OUTPUT_ROOT}',
        'wandb.mode=disabled',
        f'hydra.run.dir={OUTPUT_ROOT}/{run_tag}/hydra',
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - t0
    status  = 'ok' if result.returncode == 0 else f'err({result.returncode})'

    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(f'--- STDERR ({run_tag}) ---')
        print(result.stderr[-3000:] if result.stderr else '(no stderr)')
        crash = Path(OUTPUT_ROOT) / run_tag / 'hydra' / 'crash_report.txt'
        if crash.exists():
            print('--- crash_report.txt ---')
            print(crash.read_text()[-2000:])
        print('--- breaking after first failure ---')
        break

    with manifest_path.open('a') as f:
        f.write(f'{tilt},{seed},{status},{time.strftime("%Y-%m-%dT%H:%M:%S")},{elapsed:.0f}\n')
    print(f'  {run_tag}  {status}  ({elapsed/60:.1f} min)')

print('\nAblation loop complete.')


In [ ]:
# ── Cell 8: Results table (tilt × seed matrix) ───────────────────────────
import pandas as pd, os
from pathlib import Path

rows = []
for csv_path in sorted(Path(OUTPUT_ROOT).glob('*/metrics.csv')):
    df = pd.read_csv(csv_path)
    rows.append({
        'tilt'      : df['tilt'].iloc[0],
        'seed'      : int(df['seed'].iloc[0]),
        'final_loss': round(df['loss'].iloc[-1], 5),
        'epochs'    : len(df),
    })

if rows:
    df_all = pd.DataFrame(rows)
    pivot  = df_all.pivot(index='tilt', columns='seed', values='final_loss')
    pivot.columns = [f'seed_{c}' for c in pivot.columns]
    pivot['mean'] = pivot.mean(axis=1).round(5)
    pivot['std']  = df_all.groupby('tilt')['final_loss'].std().round(5).values
    print(pivot.to_string())
    out = os.path.join(OUTPUT_ROOT, 'results_table.csv')
    pivot.to_csv(out)
    print(f'\nSaved → {out}')
else:
    print('No completed runs yet.')


In [ ]:
# ── Cell 9: Jensen verification ───────────────────────────────────────────
import pandas as pd, numpy as np
from pathlib import Path

final_losses = {}
for csv_path in sorted(Path(OUTPUT_ROOT).glob('*/metrics.csv')):
    df   = pd.read_csv(csv_path)
    tilt = float(df['tilt'].iloc[0])
    final_losses.setdefault(tilt, []).append(df['loss'].iloc[-1])

if final_losses:
    sorted_tilts = sorted(final_losses)
    print('Tilt → mean final loss:')
    for t in sorted_tilts:
        print(f'  τ={t:+5.1f}  {np.mean(final_losses[t]):.5f}')
    if 0.0 in final_losses:
        erm = np.mean(final_losses[0.0])
        ok  = all(np.mean(final_losses[t]) >= erm - 1e-3 for t in sorted_tilts if t > 0)
        print(f'\nJensen (τ>0 ≥ ERM={erm:.5f}): {"PASS" if ok else "FAIL"}')
else:
    print('No completed runs yet.')


In [ ]:
# ── Cell 10: Plots ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import pandas as pd, numpy as np, os
from pathlib import Path

TILT_COLORS = {
    -5: '#053061', -2: '#2166ac', -1: '#74add1',
     0: '#888888',
     1: '#f46d43',  2: '#d73027',  5: '#a50026', 10: '#67001f',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

## Variant A: loss curves by tilt (mean ± std over seeds)
ax = axes[0]
tilt_dfs = {}
for csv_path in sorted(Path(OUTPUT_ROOT).glob('*/metrics.csv')):
    df   = pd.read_csv(csv_path)
    tilt = float(df['tilt'].iloc[0])
    tilt_dfs.setdefault(tilt, []).append(df[['epoch', 'loss']].set_index('epoch'))

for tilt in sorted(tilt_dfs):
    combined = pd.concat(tilt_dfs[tilt], axis=1)
    mean  = combined.mean(axis=1)
    std   = combined.std(axis=1).fillna(0)
    color = TILT_COLORS.get(tilt, 'black')
    ax.plot(mean.index, mean.values, color=color, label=f'τ={tilt:+.0f}', linewidth=1.8)
    ax.fill_between(mean.index, mean - std, mean + std, color=color, alpha=0.15)

ax.set_xlabel('Epoch'); ax.set_ylabel('NLL')
ax.set_title('Loss curves by tilt (mean ± std, 3 seeds)')
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)

## Variant B: tail QED enrichment by tilt
ax = axes[1]
rows = []
for csv_path in sorted(Path(OUTPUT_ROOT).glob('*/metrics.csv')):
    df = pd.read_csv(csv_path).dropna(subset=['qed_tail_p10'])
    if len(df):
        rows.append({'tilt': float(df['tilt'].iloc[0]), 'qed_tail': df['qed_tail_p10'].max()})

if rows:
    rd     = pd.DataFrame(rows).groupby('tilt')['qed_tail'].agg(['mean', 'std']).reset_index()
    colors = [TILT_COLORS.get(t, 'black') for t in rd['tilt']]
    ax.bar(rd['tilt'].astype(str), rd['mean'], yerr=rd['std'], color=colors, alpha=0.8, capsize=4)
    ax.axhline(0.1, color='k', linestyle='--', linewidth=1, label='Baseline 10%')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No QED data yet', ha='center', va='center',
            transform=ax.transAxes, fontsize=10, color='gray')

ax.set_xlabel('Tilt (τ)'); ax.set_ylabel('Top-10% QED fraction')
ax.set_title('Tail QED enrichment by tilt')

plt.tight_layout()
plot_path = os.path.join(OUTPUT_ROOT, 'ablation_plots.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {plot_path}')
